Score-Based Generative Modeling through Stochastic Differential Equations（ICLR 2021）  
作者：Yang Song、Jascha Sohl-Dickstein、Diederik P. Kingma、Abhishek Kumar、Stefano Ermon、Ben Poole  
[arXiv](https://arxiv.org/html/2011.13456?utm_source=chatgpt.com)

# Song et al.（2021）SDE 论文完整学习笔记

## 1. 论文研究的核心问题

先回顾我们已经学过的两篇论文。

NCSN（2019）构造多个噪声尺度：

$$
\sigma_1<\sigma_2<\cdots<\sigma_N
$$

对每个尺度学习：

$$
s_\theta(x,\sigma_i)\approx\nabla_x\log p_{\sigma_i}(x)
$$

随后使用 Annealed Langevin Dynamics 采样。

DDPM（2020）则定义：

$$
x_i=\sqrt{1-\beta_i}x_{i-1}+\sqrt{\beta_i}\epsilon_i
$$

训练噪声预测网络，并通过反向马尔可夫链采样。

表面上看，两者存在明显差异：

| 对比 | NCSN | DDPM |
|---|---|---|
| 前向扰动 | 加性高斯噪声 | 信号衰减 + 高斯噪声 |
| 训练理论 | DSM | ELBO、KL、DSM |
| 网络预测 | Score | 噪声 |
| 采样 | Annealed Langevin | 反向马尔可夫链 |

Song 等人提出：

**能否用统一的连续时间随机过程描述两种方法？**

答案是可以。

将离散时间：

$$
i=0,1,\ldots,N
$$

推广为连续时间：

$$
t\in[0,T]
$$

然后用一个随机微分方程描述前向加噪：

$$
\boxed{
dX_t=f(X_t,t)\,dt+g(t)\,dW_t
}
\tag{5}
$$

其中：

- $X_t\in\mathbb R^d$：时间 $t$ 的随机状态。
- $f:\mathbb R^d\times[0,T]\to\mathbb R^d$：漂移函数。
- $g:[0,T]\to\mathbb R_{\geq0}$：扩散系数。
- $W_t$：$d$ 维标准布朗运动。
- $p_t(x)$：$X_t$ 的边缘概率密度。

设计 $f,g$，使：

$$
X_0\sim p_{\mathrm{data}}
$$

最终：

$$
X_T\sim p_T
$$

其中 $p_T$ 是容易采样的先验，例如标准高斯。

接下来反转这个随机过程，实现：

$$
X_T\longrightarrow X_0
$$

整个论文最核心的结果是：

$$
\boxed{
dX_t=
\left[
f(X_t,t)-g(t)^2\nabla\log p_t(X_t)
\right]dt
+
g(t)d\bar W_t
}
\tag{6}
$$

这是反向时间 SDE，积分方向为 $T\rightarrow0$，所以这里 $dt<0$。

它告诉我们：

**只要知道每个时刻的边缘 Score，就能从已知的前向 SDE 构造反向生成过程。**

而 Score 可以通过 DSM 学习。

---

# 第一部分：理解 SDE 所需的数学基础

## 2. 从随机变量到随机过程

之前我们研究过：

$$
X=X_0+\sigma\epsilon
$$

其中：

$$
\epsilon\sim\mathcal N(0,I)
$$

这只描述了某个噪声尺度下的随机变量。

现在我们希望构造一整个连续随机过程：

$$
\{X_t\}_{t\in[0,T]}
$$

每个时间 $t$ 都对应一个随机变量和一个边缘分布：

$$
X_t\sim p_t
$$

需要区分三个对象：

**第一，随机过程：**

$$
X_t
$$

描述随机状态随时间的演化。

**第二，边缘概率密度：**

$$
p_t(x)
$$

描述固定时间 $t$ 下，所有可能状态的概率分布。

**第三，转移概率密度：**

$$
p_{st}(x_t\mid x_s)
$$

描述已知时间 $s$ 的状态后，时间 $t$ 的状态分布。

这里：

$$
0\leq s<t\leq T
$$

一般情况下：

$$
p_t(x_t)
=
\int p_{st}(x_t\mid x_s)p_s(x_s)\,dx_s
$$

这是 Chapman–Kolmogorov 关系的边缘化形式。

它与我们之前学习的高斯卷积具有相同的概率思想。

---

## 3. 布朗运动为什么会出现 $\sqrt{dt}$？

标准布朗运动满足：

$$
W_0=0
$$

并且对于：

$$
0\leq s<t
$$

有：

$$
W_t-W_s\sim\mathcal N(0,(t-s)I)
$$

因此，如果：

$$
h=t-s
$$

则：

$$
W_t-W_s=\sqrt h\,\epsilon
$$

其中：

$$
\epsilon\sim\mathcal N(0,I)
$$

所以：

$$
\boxed{
dW_t\sim\mathcal N(0,dt\,I)
}
$$

这里是一种微分记号，不能把 $dW_t$ 当成普通可微函数的微分。

于是 SDE：

$$
dX_t=f(X_t,t)dt+g(t)dW_t
$$

在有限小步长 $h$ 下可以近似为：

$$
\boxed{
X_{t+h}
=
X_t+f(X_t,t)h+g(t)\sqrt h\,\epsilon
}
$$

这就是 Euler–Maruyama 方法。

注意漂移和随机项的尺度不同：

$$
\text{漂移： }O(h)
$$

$$
\text{随机项： }O(\sqrt h)
$$

当 $h\rightarrow0$ 时：

$$
\sqrt h\gg h
$$

这也是随机微积分不能简单照搬普通微积分的原因。

---

## 4. Itô 引理

为了推导概率密度的演化，需要先理解 Itô 引理。

考虑一维 SDE：

$$
dX_t=f(X_t,t)dt+g(t)dW_t
$$

设：

$$
\phi(x,t)
$$

是足够光滑的函数。

普通微积分中的二阶泰勒展开为：

$$
\begin{aligned}
d\phi
\approx{}&
\frac{\partial\phi}{\partial t}dt
+
\frac{\partial\phi}{\partial x}dX_t\\
&+
\frac12
\frac{\partial^2\phi}{\partial x^2}
(dX_t)^2
\end{aligned}
$$

在普通 ODE 中，通常可以忽略：

$$
(dX_t)^2
$$

但 SDE 不行。

因为：

$$
dX_t=fdt+gdW_t
$$

所以：

$$
(dX_t)^2
=
f^2dt^2+2fg\,dt\,dW_t+g^2(dW_t)^2
$$

Itô 微积分中的二次变差规则是：

$$
(dt)^2=0
$$

$$
dt\,dW_t=0
$$

$$
(dW_t)^2=dt
$$

这里表示在 Itô 微积分的极限意义下保留的项。

因此：

$$
\boxed{
(dX_t)^2=g(t)^2dt
}
$$

代入泰勒展开：

$$
\boxed{
d\phi(X_t,t)
=
\left[
\frac{\partial\phi}{\partial t}
+
f\frac{\partial\phi}{\partial x}
+
\frac12g^2\frac{\partial^2\phi}{\partial x^2}
\right]dt
+
g\frac{\partial\phi}{\partial x}dW_t
}
$$

推广到 $d$ 维：

$$
\boxed{
\begin{aligned}
d\phi(X_t,t)
={}&
\left[
\partial_t\phi
+
f^\top\nabla\phi
+
\frac12g^2\Delta\phi
\right]dt\\
&+
g\nabla\phi^\top dW_t
\end{aligned}
}
$$

其中：

$$
\Delta\phi
=
\sum_{i=1}^d
\frac{\partial^2\phi}{\partial x_i^2}
$$

是拉普拉斯算子。

**Itô 引理最重要的区别，就是多出了二阶导数项。**

---

## 5. 从 Itô 引理推导 Fokker–Planck 方程

现在考虑：

$$
dX_t=f(X_t,t)dt+g(t)dW_t
$$

我们想知道：

$$
p_t(x)
$$

如何随时间变化。

取一个不显含时间的光滑测试函数：

$$
\phi(x)
$$

根据 Itô 引理：

$$
d\phi(X_t)
=
\left[
f^\top\nabla\phi+
\frac12g^2\Delta\phi
\right]dt
+
g\nabla\phi^\top dW_t
$$

两边取期望。

由于 Itô 积分的期望为零：

$$
\mathbb E\left[
\int g\nabla\phi^\top dW_t
\right]=0
$$

因此：

$$
\boxed{
\frac{d}{dt}\mathbb E[\phi(X_t)]
=
\mathbb E
\left[
f^\top\nabla\phi+
\frac12g^2\Delta\phi
\right]
}
$$

利用概率密度表示期望：

$$
\mathbb E[\phi(X_t)]
=
\int\phi(x)p_t(x)\,dx
$$

于是左边：

$$
\frac{d}{dt}\mathbb E[\phi(X_t)]
=
\int\phi(x)\partial_tp_t(x)\,dx
$$

右边：

$$
\int f^\top\nabla\phi\,p_t\,dx
+
\frac12g^2\int\Delta\phi\,p_t\,dx
$$

对第一项分部积分：

$$
\int f^\top\nabla\phi\,p_t\,dx
=
-\int\phi\nabla\cdot(fp_t)\,dx
$$

对第二项分部积分两次：

$$
\int\Delta\phi\,p_t\,dx
=
\int\phi\Delta p_t\,dx
$$

这里假设边界项消失，例如密度和测试函数具有足够快的衰减。

因此：

$$
\int\phi\partial_tp_t\,dx
=
\int\phi
\left[
-\nabla\cdot(fp_t)+\frac12g^2\Delta p_t
\right]dx
$$

由于这个等式对任意合适的测试函数 $\phi$ 成立，得到：

$$
\boxed{
\frac{\partial p_t}{\partial t}
=
-\nabla\cdot(fp_t)
+
\frac12g(t)^2\Delta p_t
}
\tag{FP}
$$

这就是 Fokker–Planck 方程。

它告诉我们：

- 漂移 $f$ 负责运输概率质量。
- 扩散 $g$ 负责平滑概率密度。

后面推导 Probability Flow ODE 时，我们会再次使用它。

---

# 第二部分：VE、VP、sub-VP SDE 的完整推导

## 6. VE-SDE：从 NCSN 推导连续时间过程

NCSN 使用：

$$
X_{\sigma}=X_0+\sigma\epsilon
$$

设离散噪声尺度：

$$
0=\sigma_0<\sigma_1<\cdots<\sigma_N
$$

构造马尔可夫链：

$$
\boxed{
X_i=X_{i-1}
+
\sqrt{\sigma_i^2-\sigma_{i-1}^2}\epsilon_i
}
$$

其中：

$$
\epsilon_i\sim\mathcal N(0,I)
$$

为什么是方差之差？

因为：

$$
X_i-X_0
=
\sum_{j=1}^i
\sqrt{\sigma_j^2-\sigma_{j-1}^2}\epsilon_j
$$

各个噪声独立，所以累计方差为：

$$
\begin{aligned}
\operatorname{Var}(X_i-X_0)
&=
\sum_{j=1}^i
(\sigma_j^2-\sigma_{j-1}^2)I\\
&=\sigma_i^2I
\end{aligned}
$$

因此：

$$
\boxed{
X_i\mid X_0
\sim\mathcal N(X_0,\sigma_i^2I)
}
$$

现在令：

$$
t_i=i\Delta t
$$

并使：

$$
\sigma_i=\sigma(t_i)
$$

当：

$$
\Delta t\rightarrow0
$$

时：

$$
\sigma^2(t+\Delta t)-\sigma^2(t)
\approx
\frac{d\sigma^2(t)}{dt}\Delta t
$$

于是：

$$
X_{t+\Delta t}-X_t
\approx
\sqrt{
\frac{d\sigma^2(t)}{dt}
}\sqrt{\Delta t}\epsilon
$$

因此：

$$
\boxed{
dX_t=
\sqrt{
\frac{d\sigma^2(t)}{dt}
}\,dW_t
}
\tag{9}
$$

这就是 VE-SDE。

其中：

$$
f(x,t)=0
$$

$$
g(t)^2=\frac{d\sigma^2(t)}{dt}
$$

### 6.1 VE-SDE 的解析解

积分：

$$
X_t=X_0+\int_0^t g(s)dW_s
$$

根据 Itô 等距性质：

$$
\operatorname{Cov}
\left[
\int_0^t g(s)dW_s
\right]
=
\int_0^t g(s)^2ds\,I
$$

所以：

$$
\operatorname{Cov}(X_t\mid X_0)
=
[\sigma^2(t)-\sigma^2(0)]I
$$

因此：

$$
\boxed{
p_{0t}(x_t\mid x_0)
=
\mathcal N
\left(
x_t;x_0,
[\sigma^2(t)-\sigma^2(0)]I
\right)
}
$$

如果从零噪声起点开始：

$$
\sigma(0)=0
$$

就有：

$$
\boxed{
X_t=X_0+\sigma(t)\epsilon
}
$$

这与 NCSN 完全对应。

### 6.2 论文中的指数噪声调度

论文采用：

$$
\boxed{
\sigma(t)
=
\sigma_{\min}
\left(
\frac{\sigma_{\max}}{\sigma_{\min}}
\right)^t
}
$$

定义：

$$
r=\frac{\sigma_{\max}}{\sigma_{\min}}
$$

那么：

$$
\sigma(t)=\sigma_{\min}r^t
$$

所以：

$$
\sigma^2(t)=\sigma_{\min}^2r^{2t}
$$

求导：

$$
\frac{d\sigma^2(t)}{dt}
=
2\log(r)\sigma_{\min}^2r^{2t}
$$

因此：

$$
\boxed{
g(t)
=
\sigma_{\min}r^t\sqrt{2\log r}
}
$$

得到论文附录 C 的具体 VE-SDE：

$$
\boxed{
dX_t=
\sigma_{\min}
\left(
\frac{\sigma_{\max}}{\sigma_{\min}}
\right)^t
\sqrt{
2\log\frac{\sigma_{\max}}{\sigma_{\min}}
}
\,dW_t
}
$$

**一个细节：** 论文采用的离散到连续的约定，在 $t=0$ 存在噪声尺度不连续的问题：原始数据对应零噪声，但 $t\to0^+$ 的噪声尺度趋近于 $\sigma_{\min}$。所以论文实际在 $t\in[\varepsilon,1]$ 上求解，并使用相应的非零噪声扰动核。不能同时把上述光滑指数 $\sigma(t)$ 的 $\sigma(0)=\sigma_{\min}$ 和零噪声初始条件直接代入而不作说明。[arXiv](https://arxiv.org/html/2011.13456?utm_source=chatgpt.com)

---

## 7. VP-SDE：从 DDPM 推导连续时间过程

DDPM 的离散前向过程为：

$$
\boxed{
X_i=
\sqrt{1-\beta_i}X_{i-1}
+
\sqrt{\beta_i}\epsilon_i
}
$$

为了构造连续极限，不能简单地令：

$$
\beta_i\rightarrow\beta(t)
$$

而应当将离散噪声方差与时间步长联系起来：

$$
\boxed{
\beta_i=\beta(t_i)\Delta t
}
$$

因此：

$$
X_{t+\Delta t}
=
\sqrt{1-\beta(t)\Delta t}X_t
+
\sqrt{\beta(t)\Delta t}\epsilon
$$

对平方根进行泰勒展开：

$$
\sqrt{1-z}
=
1-\frac z2+O(z^2)
$$

取：

$$
z=\beta(t)\Delta t
$$

得到：

$$
\sqrt{1-\beta(t)\Delta t}
=
1-\frac12\beta(t)\Delta t+O(\Delta t^2)
$$

所以：

$$
X_{t+\Delta t}-X_t
=
-\frac12\beta(t)X_t\Delta t
+
\sqrt{\beta(t)\Delta t}\epsilon
+
O(\Delta t^2)
$$

令：

$$
\Delta t\rightarrow0
$$

得到：

$$
\boxed{
dX_t=
-\frac12\beta(t)X_tdt+
\sqrt{\beta(t)}dW_t
}
\tag{11}
$$

这就是 VP-SDE。

---

## 8. VP-SDE 条件分布的完整求解

考虑：

$$
dX_t=
-\frac12\beta(t)X_tdt+
\sqrt{\beta(t)}dW_t
$$

定义累计噪声强度：

$$
\boxed{
B(t)=\int_0^t\beta(s)ds
}
$$

令：

$$
A(t)=e^{-B(t)/2}
$$

我们希望得到：

$$
p_{0t}(x_t\mid x_0)
$$

### 8.1 使用积分因子

定义：

$$
Y_t=e^{B(t)/2}X_t
$$

因为：

$$
d[e^{B(t)/2}]
=
\frac12\beta(t)e^{B(t)/2}dt
$$

使用 Itô 乘积法则：

$$
dY_t
=
e^{B(t)/2}dX_t+
X_td[e^{B(t)/2}]
$$

代入 SDE：

$$
\begin{aligned}
dY_t
={}&
e^{B(t)/2}
\left[
-\frac12\beta(t)X_tdt+
\sqrt{\beta(t)}dW_t
\right]\\
&+
\frac12\beta(t)e^{B(t)/2}X_tdt
\end{aligned}
$$

漂移项抵消：

$$
\boxed{
dY_t=e^{B(t)/2}\sqrt{\beta(t)}dW_t
}
$$

积分：

$$
Y_t=X_0+
\int_0^t e^{B(s)/2}\sqrt{\beta(s)}dW_s
$$

因此：

$$
\boxed{
X_t=
e^{-B(t)/2}X_0+
e^{-B(t)/2}
\int_0^t e^{B(s)/2}\sqrt{\beta(s)}dW_s
}
$$

### 8.2 求条件均值

由于 Itô 积分均值为零：

$$
\mathbb E[X_t\mid X_0=x_0]
=
e^{-B(t)/2}x_0
$$

所以：

$$
\boxed{
m_t(x_0)=e^{-B(t)/2}x_0
}
$$

### 8.3 求条件方差

随机项为：

$$
Z_t=
e^{-B(t)/2}
\int_0^t e^{B(s)/2}\sqrt{\beta(s)}dW_s
$$

根据 Itô 等距：

$$
\operatorname{Cov}(Z_t)
=
e^{-B(t)}
\int_0^t e^{B(s)}\beta(s)ds\,I
$$

注意：

$$
\frac{d}{ds}e^{B(s)}
=
e^{B(s)}\beta(s)
$$

因此：

$$
\int_0^t e^{B(s)}\beta(s)ds
=
e^{B(t)}-1
$$

所以：

$$
\begin{aligned}
\operatorname{Cov}(Z_t)
&=
e^{-B(t)}[e^{B(t)}-1]I\\
&=
[1-e^{-B(t)}]I
\end{aligned}
$$

最终：

$$
\boxed{
p_{0t}(x_t\mid x_0)
=
\mathcal N
\left(
x_t;
e^{-B(t)/2}x_0,
[1-e^{-B(t)}]I
\right)
}
$$

等价于：

$$
\boxed{
X_t=
e^{-B(t)/2}X_0+
\sqrt{1-e^{-B(t)}}\epsilon
}
$$

其中：

$$
\epsilon\sim\mathcal N(0,I)
$$

对比 DDPM：

$$
X_t=
\sqrt{\bar\alpha_t}X_0+
\sqrt{1-\bar\alpha_t}\epsilon
$$

可以发现：

$$
\boxed{
\bar\alpha(t)=e^{-B(t)}
}
$$

这就是 VP-SDE 与 DDPM 的连续时间对应关系。

---

## 9. 为什么称为 Variance Preserving？

这里的 Preserving 不是说每条样本轨迹的方差都不变，也不是说任意初始分布的方差始终不变。

它的准确含义是：

**如果初始协方差为单位矩阵，那么 VP-SDE 在整个扩散过程中保持单位协方差。**

下面推导。

设：

$$
\Sigma_t=\operatorname{Cov}(X_t)
$$

由解析解：

$$
X_t=e^{-B(t)/2}X_0+\sqrt{1-e^{-B(t)}}\epsilon
$$

因为 $X_0$ 与 $\epsilon$ 独立：

$$
\Sigma_t
=
e^{-B(t)}\Sigma_0+
[1-e^{-B(t)}]I
$$

整理：

$$
\boxed{
\Sigma_t
=
I+e^{-B(t)}(\Sigma_0-I)
}
$$

如果：

$$
\Sigma_0=I
$$

那么：

$$
\boxed{
\Sigma_t=I
}
$$

对任意 $t$ 成立。

这也是论文附录 B 的公式 (26)。[arXiv](https://arxiv.org/html/2011.13456?utm_source=chatgpt.com)

但如果：

$$
\Sigma_0\neq I
$$

那么协方差一般会随时间变化，只是在：

$$
B(t)\rightarrow\infty
$$

时趋近：

$$
I
$$

---

## 10. sub-VP SDE：论文新增的第三种扩散过程

除了 VE 和 VP，论文还提出 sub-VP SDE：

$$
\boxed{
dX_t=
-\frac12\beta(t)X_tdt+
\sqrt{
\beta(t)\left[1-e^{-2B(t)}\right]
}
\,dW_t
}
\tag{12}
$$

其中：

$$
B(t)=\int_0^t\beta(s)ds
$$

它与 VP 的漂移完全相同：

$$
f(x,t)=-\frac12\beta(t)x
$$

但扩散系数不同。

VP：

$$
g_{\mathrm{VP}}^2(t)=\beta(t)
$$

sub-VP：

$$
\boxed{
g_{\mathrm{sub}}^2(t)
=
\beta(t)[1-e^{-2B(t)}]
}
$$

由于：

$$
0\leq1-e^{-2B(t)}\leq1
$$

所以：

$$
\boxed{
g_{\mathrm{sub}}^2(t)\leq g_{\mathrm{VP}}^2(t)
}
$$

这说明 sub-VP 在相同时间点的瞬时噪声注入强度不超过 VP。

但我们还需要进一步证明它的累计条件方差也更小。

### 10.1 求解 sub-VP 的条件均值

因为漂移与 VP 相同，所以积分因子也相同。

得到：

$$
\boxed{
\mathbb E[X_t\mid X_0=x_0]
=
e^{-B(t)/2}x_0
}
$$

### 10.2 求解条件方差

sub-VP 的随机积分项为：

$$
Z_t=
e^{-B(t)/2}
\int_0^t
e^{B(s)/2}
\sqrt{
\beta(s)[1-e^{-2B(s)}]
}
\,dW_s
$$

所以：

$$
\begin{aligned}
\operatorname{Var}(Z_t)
={}&
e^{-B(t)}
\int_0^t
e^{B(s)}
\beta(s)[1-e^{-2B(s)}]ds
\end{aligned}
$$

展开：

$$
=
e^{-B(t)}
\int_0^t
\beta(s)[e^{B(s)}-e^{-B(s)}]ds
$$

分别积分。

第一项：

$$
\int_0^t\beta(s)e^{B(s)}ds
=
e^{B(t)}-1
$$

第二项：

$$
\int_0^t\beta(s)e^{-B(s)}ds
=
1-e^{-B(t)}
$$

因此：

$$
\begin{aligned}
\operatorname{Var}(Z_t)
&=
e^{-B(t)}
\left[
e^{B(t)}-1-(1-e^{-B(t)})
\right]\\
&=
e^{-B(t)}
[e^{B(t)}-2+e^{-B(t)}]\\
&=
1-2e^{-B(t)}+e^{-2B(t)}
\end{aligned}
$$

所以：

$$
\boxed{
\operatorname{Var}(Z_t)
=
[1-e^{-B(t)}]^2
}
$$

最终：

$$
\boxed{
p_{0t}^{\mathrm{sub}}(x_t\mid x_0)
=
\mathcal N
\left(
e^{-B(t)/2}x_0,
[1-e^{-B(t)}]^2I
\right)
}
$$

对比 VP：

$$
\operatorname{Var}_{\mathrm{VP}}(X_t\mid X_0)
=
1-e^{-B(t)}
$$

sub-VP：

$$
\operatorname{Var}_{\mathrm{sub}}(X_t\mid X_0)
=
[1-e^{-B(t)}]^2
$$

因为：

$$
0\leq1-e^{-B(t)}\leq1
$$

所以：

$$
\boxed{
[1-e^{-B(t)}]^2
\leq
1-e^{-B(t)}
}
$$

因此：

$$
\boxed{
\operatorname{Var}_{\mathrm{sub}}(X_t\mid X_0)
\leq
\operatorname{Var}_{\mathrm{VP}}(X_t\mid X_0)
}
$$

这是 sub-VP 名称的数学依据。

### 10.3 为什么 sub-VP 有助于似然？

论文实验发现，sub-VP 在 Probability Flow ODE 的似然评估上优于对应的 VP 设置。

但应当注意：

**较小的条件方差并不能单独从数学上推出更好的似然。**

实际似然还受到网络表达能力、训练权重、噪声调度以及 ODE 数值误差等因素影响。

论文报告的是经验优势，而不是“条件方差越小，似然必然越高”的定理。

---

# 第三部分：反向时间 SDE 的严格理解

## 11. 为什么前向过程能够反转？

现在我们已经构造了：

$$
dX_t=f(X_t,t)dt+g(t)dW_t
$$

前向过程把复杂数据逐渐变成简单噪声。

但生成模型需要反过来：

$$
p_T\longrightarrow p_0
$$

一个容易产生的错误想法是：

> 既然前向过程的漂移是 $f$，那么反向过程是不是直接把漂移改成 $-f$？

答案是否定的。

原因是随机过程的反向演化不仅依赖局部漂移，还依赖当前状态的概率密度。

例如纯布朗运动：

$$
dX_t=dW_t
$$

前向漂移为零。

但如果从终点反向恢复原始数据，反向漂移通常不为零。

因为反向过程必须知道：

**哪些状态更可能来自原始数据分布。**

这个信息恰好由 Score 提供：

$$
\nabla_x\log p_t(x)
$$

---

## 12. 反向 SDE 的公式

论文引用 Anderson（1982）的时间反转结果：

$$
\boxed{
dX_t=
\left[
f(X_t,t)-g(t)^2\nabla\log p_t(X_t)
\right]dt
+
g(t)d\bar W_t
}
\tag{6}
$$

注意：

$$
dt<0
$$

为了避免负时间步导致符号混乱，我们可以定义新的正向时间：

$$
\boxed{
\tau=T-t
}
$$

令：

$$
Y_\tau=X_{T-\tau}
$$

那么：

$$
\tau:0\rightarrow T
$$

对应原始扩散时间：

$$
t:T\rightarrow0
$$

此时反向 SDE 可以写为：

$$
\boxed{
dY_\tau=
\left[
-f(Y_\tau,T-\tau)
+
g(T-\tau)^2\nabla\log p_{T-\tau}(Y_\tau)
\right]d\tau
+
g(T-\tau)d\widetilde W_\tau
}
$$

这个形式更加直观：

- 前向漂移取负。
- 增加 Score 引导。
- 保留随机扩散。

---

## 13. 从贝叶斯公式推导反向漂移

这里先推导标量扩散系数 $g(t)$ 且与状态 $x$ 无关的情况。

取一个很小的正时间间隔：

$$
h>0
$$

我们希望计算：

$$
p(X_{t-h}=y\mid X_t=x)
$$

根据贝叶斯公式：

$$
\boxed{
p(y\mid x)
=
\frac{
p(x\mid y)p_{t-h}(y)
}{
p_t(x)
}
}
$$

其中：

$$
p(x\mid y)
$$

是前向短时间转移密度。

对于小 $h$，前向 SDE 给出：

$$
X_t
\approx
X_{t-h}
+
f(X_{t-h},t-h)h
+
g(t)\sqrt h\,\epsilon
$$

因此：

$$
\boxed{
p(x\mid y)
\approx
\mathcal N
\left(
x;y+f(y,t)h,g(t)^2hI
\right)
}
$$

为了求反向均值，令：

$$
y=x+\delta
$$

其中主要概率质量集中在：

$$
\delta=O(\sqrt h)
$$

附近。

根据贝叶斯公式：

$$
\log p(y\mid x)
=
\log p(x\mid y)
+
\log p_{t-h}(y)
-
\log p_t(x)
$$

最后一项不依赖 $y$，可以并入归一化常数。

### 13.1 展开前向转移密度

$$
\log p(x\mid y)
=
-\frac{
\|x-y-f(y,t)h\|^2
}{
2g(t)^2h
}
+C
$$

代入：

$$
y=x+\delta
$$

并在一阶漂移精度下近似：

$$
f(y,t)\approx f(x,t)
$$

得到：

$$
\log p(x\mid y)
\approx
-\frac{
\|\delta+f(x,t)h\|^2
}{
2g(t)^2h
}
+C
$$

展开平方：

$$
\|\delta+fh\|^2
=
\|\delta\|^2+2h\delta^\top f+h^2\|f\|^2
$$

所以：

$$
\log p(x\mid y)
\approx
-\frac{\|\delta\|^2}{2g^2h}
-
\frac{\delta^\top f}{g^2}
+
C+O(h)
$$

### 13.2 展开边缘概率密度

对：

$$
\log p_{t-h}(x+\delta)
$$

做泰勒展开：

$$
\log p_{t-h}(x+\delta)
=
\log p_t(x)
+
\delta^\top\nabla\log p_t(x)
+
O(h,\|\delta\|^2)
$$

因此：

$$
\begin{aligned}
\log p(y\mid x)
\approx{}&
-\frac{\|\delta\|^2}{2g^2h}\\
&+
\delta^\top
\left[
\nabla\log p_t(x)-\frac{f(x,t)}{g^2}
\right]
+C
\end{aligned}
$$

### 13.3 配方

定义：

$$
a(x,t)=g(t)^2\nabla\log p_t(x)-f(x,t)
$$

那么：

$$
\log p(y\mid x)
\approx
-\frac{\|\delta\|^2}{2g^2h}
+
\frac{\delta^\top a}{g^2}
+C
$$

配方：

$$
\boxed{
\log p(y\mid x)
\approx
-\frac{
\|\delta-ha\|^2
}{
2g^2h
}
+C'
}
$$

因此：

$$
\boxed{
X_{t-h}\mid X_t=x
\approx
\mathcal N
\left(
x+h[g^2\nabla\log p_t(x)-f(x,t)],
g^2hI
\right)
}
$$

于是：

$$
\boxed{
X_{t-h}
=
X_t+
h[g(t)^2\nabla\log p_t(X_t)-f(X_t,t)]
+
g(t)\sqrt h\,\epsilon
}
$$

这是反向 SDE 的 Euler–Maruyama 形式。

令：

$$
dt=-h
$$

便得到：

$$
\boxed{
dX_t=
[f-g^2\nabla\log p_t]dt+
g\,d\bar W_t
}
$$

严格的时间反转定理还需要对密度光滑性、扩散系数和边界行为作出条件限制。上面的局部展开主要解释 Score 项的来源，并非替代完整的随机过程时间反转定理。

---

## 14. 用一个高斯例子验证反向 SDE

考虑最简单的前向过程：

$$
dX_t=dW_t
$$

并令：

$$
X_0=0
$$

那么：

$$
X_t=W_t
$$

所以：

$$
p_t(x)=\mathcal N(0,t)
$$

其 Score 为：

$$
\boxed{
\nabla_x\log p_t(x)=-\frac{x}{t}
}
$$

因为：

$$
f=0,\qquad g=1
$$

所以反向 SDE 为：

$$
dX_t=
-\nabla\log p_t(X_t)dt+d\bar W_t
$$

代入 Score：

$$
\boxed{
dX_t=\frac{X_t}{t}dt+d\bar W_t
}
$$

注意反向时间：

$$
dt=-h
$$

因此：

$$
X_{t-h}
\approx
X_t-\frac{h}{t}X_t+\sqrt h\,\epsilon
$$

也就是：

$$
\boxed{
X_{t-h}
\approx
\left(1-\frac ht\right)X_t+\sqrt h\,\epsilon
}
$$

反向漂移会把状态逐渐拉回零。

这符合直觉：因为前向过程从确定的原点出发，反向过程最终也应该回到原点。

更精确地，布朗运动的条件分布为：

$$
\boxed{
X_{t-h}\mid X_t=x
\sim
\mathcal N
\left(
\frac{t-h}{t}x,
\frac{h(t-h)}{t}
\right)
}
$$

当 $h$ 很小时：

$$
\frac{h(t-h)}t=h+O(h^2)
$$

与上面的反向 SDE 局部近似一致。

这个例子可以帮助我们理解：

**反向 Score 并不是额外人为添加的去噪技巧，而是反转随机扩散过程时自然出现的概率密度修正项。**

---

# 第四部分：连续时间 DSM 的完整推导

## 15. 为什么训练时只需要前向条件分布？

论文定义：

$$
\boxed{
\begin{aligned}
\mathcal L(\theta)
=
\mathbb E_t\Big[
\lambda(t)
\mathbb E_{X_0}
\mathbb E_{X_t\mid X_0}
\|
s_\theta(X_t,t)
-
\nabla_{X_t}\log p_{0t}(X_t\mid X_0)
\|^2
\Big]
\end{aligned}
}
\tag{7}
$$

其中：

$$
t\sim\operatorname{Uniform}[\varepsilon,T]
$$

实际实现中通常避免在零噪声端点直接训练。

这里的关键问题与 NCSN 完全相同：

> 网络使用条件 Score 作为监督，为什么最终学习到的是边缘 Score？

我们重新从条件期望角度推导。

定义条件 Score：

$$
u(X_t,X_0,t)
=
\nabla_{X_t}\log p_{0t}(X_t\mid X_0)
$$

真实边缘 Score：

$$
s^*(X_t,t)=\nabla\log p_t(X_t)
$$

根据边缘化：

$$
p_t(x)
=
\int p_{0t}(x\mid x_0)p_0(x_0)\,dx_0
$$

对 $x$ 求梯度：

$$
\nabla p_t(x)
=
\int
\nabla p_{0t}(x\mid x_0)p_0(x_0)\,dx_0
$$

利用：

$$
\nabla p_{0t}
=
p_{0t}\nabla\log p_{0t}
$$

得到：

$$
\nabla p_t(x)
=
\int
p_{0t}(x\mid x_0)p_0(x_0)
\nabla\log p_{0t}(x\mid x_0)
\,dx_0
$$

两边除以：

$$
p_t(x)
$$

于是：

$$
\begin{aligned}
\nabla\log p_t(x)
&=
\int
\frac{
p_{0t}(x\mid x_0)p_0(x_0)
}{
p_t(x)
}
\nabla\log p_{0t}(x\mid x_0)
\,dx_0
\end{aligned}
$$

根据贝叶斯公式：

$$
p(x_0\mid x)
=
\frac{
p_{0t}(x\mid x_0)p_0(x_0)
}{
p_t(x)
}
$$

因此：

$$
\boxed{
\nabla\log p_t(x)
=
\mathbb E
\left[
\nabla\log p_{0t}(x\mid X_0)
\mid X_t=x
\right]
}
$$

这就是 DSM 恒等式。

### 15.1 为什么 MSE 学习到条件期望？

固定：

$$
X_t=x,\quad t
$$

网络的最优预测为：

$$
s^*(x,t)
=
\arg\min_s
\mathbb E[\|s-u\|^2\mid X_t=x]
$$

展开：

$$
\mathbb E[\|s-u\|^2]
=
\|s\|^2-2s^\top\mathbb E[u]+\mathbb E[\|u\|^2]
$$

对 $s$ 求导：

$$
2s-2\mathbb E[u]=0
$$

所以：

$$
\boxed{
s^*(x,t)=\mathbb E[u\mid X_t=x]
}
$$

结合前面的 DSM 恒等式：

$$
\boxed{
s_\theta^*(x,t)=\nabla\log p_t(x)
}
$$

因此：

**训练时使用的是容易计算的条件 Score，但网络在 MSE 意义下学习的是复杂边缘分布的 Score。**

---

## 16. 为什么噪声预测 MSE 与 Score Matching 等价？

对于 VE、VP 和 sub-VP，这些 SDE 的条件转移分布都具有高斯形式：

$$
\boxed{
p_{0t}(x_t\mid x_0)
=
\mathcal N(m_t(x_0),\sigma_t^2I)
}
$$

所以：

$$
X_t=m_t(X_0)+\sigma_t\epsilon
$$

其中：

$$
\epsilon\sim\mathcal N(0,I)
$$

条件 Score：

$$
\boxed{
\nabla_{x_t}\log p_{0t}(x_t\mid x_0)
=
-\frac{x_t-m_t(x_0)}{\sigma_t^2}
=
-\frac{\epsilon}{\sigma_t}
}
$$

代入 DSM：

$$
\mathcal L
=
\mathbb E
\left[
\lambda(t)
\left\|
s_\theta(X_t,t)+\frac{\epsilon}{\sigma_t}
\right\|^2
\right]
$$

如果选择：

$$
\boxed{
\lambda(t)=\sigma_t^2
}
$$

则：

$$
\boxed{
\mathcal L
=
\mathbb E
\left[
\|\sigma_ts_\theta(X_t,t)+\epsilon\|^2
\right]
}
$$

定义：

$$
\boxed{
\epsilon_\theta(X_t,t)=-\sigma_ts_\theta(X_t,t)
}
$$

就得到：

$$
\boxed{
\mathcal L
=
\mathbb E
\left[
\|\epsilon_\theta(X_t,t)-\epsilon\|^2
\right]
}
$$

因此：

$$
\boxed{
\text{连续时间 DSM}
\Longleftrightarrow
\text{噪声预测 MSE}
}
$$

这里的等价指在相应参数化与权重选择下，两个损失具有相同形式。

它并不意味着不同网络参数化、不同时间采样策略和不同优化设置必然获得相同训练结果。

---

## 17. 为什么权重选择为条件方差？

论文提出一种常见权重设计：

$$
\lambda(t)
\propto
\frac1{
\mathbb E[
\|\nabla\log p_{0t}(X_t\mid X_0)\|^2
]
}
$$

对高斯条件分布：

$$
\nabla\log p_{0t}
=
-\frac{\epsilon}{\sigma_t}
$$

所以：

$$
\mathbb E
\left[
\left\|-\frac{\epsilon}{\sigma_t}\right\|^2
\right]
=
\frac1{\sigma_t^2}
\mathbb E[\|\epsilon\|^2]
$$

由于：

$$
\epsilon\sim\mathcal N(0,I_d)
$$

有：

$$
\mathbb E[\|\epsilon\|^2]=d
$$

因此：

$$
\boxed{
\mathbb E[\|\nabla\log p_{0t}\|^2]
=
\frac d{\sigma_t^2}
}
$$

所以：

$$
\boxed{
\lambda(t)\propto\sigma_t^2
}
$$

这说明选择条件方差作为权重，可以抵消不同噪声尺度下条件 Score 的典型量级差异。

但要注意，它只是合理的权重设计原则，不是唯一选择。

---

# 第五部分：Probability Flow ODE 的完整推导

## 18. 为什么一个随机 SDE 可以对应确定性 ODE？

这是整篇论文最重要的理论创新之一。

前向 SDE：

$$
\boxed{
dX_t=f(X_t,t)dt+g(t)dW_t
}
$$

对应的 Fokker–Planck 方程：

$$
\boxed{
\partial_tp_t
=
-\nabla\cdot(fp_t)+\frac12g^2\Delta p_t
}
$$

我们的目标是寻找一个确定性 ODE：

$$
\frac{dX_t}{dt}=v(X_t,t)
$$

使它与 SDE 具有相同的边缘概率密度：

$$
p_t
$$

注意，我们要求的是：

$$
\boxed{
p_t^{\mathrm{ODE}}=p_t^{\mathrm{SDE}}
}
$$

而不是要求：

$$
X_t^{\mathrm{ODE}}=X_t^{\mathrm{SDE}}
$$

单条轨迹通常不同。

---

## 19. ODE 对应的连续性方程

考虑确定性 ODE：

$$
\frac{dX_t}{dt}=v(X_t,t)
$$

如果粒子按照速度场 $v$ 运动，概率质量守恒要求：

$$
\boxed{
\partial_tp_t=-\nabla\cdot(vp_t)
}
$$

这就是连续性方程。

为什么是这个形式？

考虑空间区域：

$$
\Omega\subset\mathbb R^d
$$

其中的概率质量：

$$
M_\Omega(t)=\int_\Omega p_t(x)dx
$$

概率质量只能通过边界流入或流出。

根据散度定理：

$$
\frac{dM_\Omega}{dt}
=
-\int_{\partial\Omega}p_tv\cdot n\,dS
$$

等价于：

$$
\frac{dM_\Omega}{dt}
=
-\int_\Omega\nabla\cdot(p_tv)\,dx
$$

另一方面：

$$
\frac{dM_\Omega}{dt}
=
\int_\Omega\partial_tp_t\,dx
$$

所以：

$$
\int_\Omega
\left[
\partial_tp_t+\nabla\cdot(p_tv)
\right]dx=0
$$

由于区域 $\Omega$ 可以任意选取：

$$
\boxed{
\partial_tp_t+\nabla\cdot(p_tv)=0
}
$$

---

## 20. 从 Fokker–Planck 推导 Probability Flow ODE

回到 SDE 的 Fokker–Planck 方程：

$$
\partial_tp_t
=
-\nabla\cdot(fp_t)
+
\frac12g^2\Delta p_t
$$

因为：

$$
\Delta p_t=\nabla\cdot(\nabla p_t)
$$

所以：

$$
\partial_tp_t
=
-\nabla\cdot(fp_t)
+
\frac12g^2\nabla\cdot(\nabla p_t)
$$

由于 $g(t)$ 不依赖 $x$，可以将它移入散度：

$$
\partial_tp_t
=
-\nabla\cdot
\left[
fp_t-\frac12g^2\nabla p_t
\right]
$$

再利用：

$$
\boxed{
\nabla p_t=p_t\nabla\log p_t
}
$$

得到：

$$
\partial_tp_t
=
-\nabla\cdot
\left[
fp_t-\frac12g^2p_t\nabla\log p_t
\right]
$$

提取 $p_t$：

$$
\boxed{
\partial_tp_t
=
-\nabla\cdot
\left[
\left(
f-\frac12g^2\nabla\log p_t
\right)p_t
\right]
}
$$

与 ODE 的连续性方程比较：

$$
\partial_tp_t=-\nabla\cdot(vp_t)
$$

因此选择：

$$
\boxed{
v(x,t)
=
f(x,t)-\frac12g(t)^2\nabla\log p_t(x)
}
$$

就得到：

$$
\boxed{
\frac{dX_t}{dt}
=
f(X_t,t)
-
\frac12g(t)^2\nabla\log p_t(X_t)
}
\tag{13}
$$

这就是 Probability Flow ODE。

### 20.1 为什么两者具有相同边缘分布？

因为它们满足同一个概率密度演化方程：

$$
\boxed{
\partial_tp_t
=
-\nabla\cdot
\left[
\left(
f-\frac12g^2\nabla\log p_t
\right)p_t
\right]
}
$$

如果初始分布相同：

$$
p_0^{\mathrm{ODE}}=p_0^{\mathrm{SDE}}
$$

并且相关演化方程的解唯一，那么：

$$
\boxed{
p_t^{\mathrm{ODE}}=p_t^{\mathrm{SDE}}
}
$$

对所有：

$$
t\in[0,T]
$$

成立。

**这不是说 SDE 中的随机噪声消失了，而是说随机扩散对概率密度造成的影响，可以通过一个依赖 Score 的确定性速度场来表示。**

---

## 21. 为什么 SDE 和 ODE 的 Score 系数相差两倍？

我们已经得到：

反向 SDE：

$$
\boxed{
dX_t=
[f-g^2s_t]dt+g\,d\bar W_t
}
$$

Probability Flow ODE：

$$
\boxed{
dX_t=
[f-\frac12g^2s_t]dt
}
$$

其中：

$$
s_t=\nabla\log p_t
$$

为什么是：

$$
g^2
$$

和：

$$
\frac12g^2
$$

而不是相同的系数？

原因就在 Fokker–Planck 方程的扩散项：

$$
\frac12g^2\Delta p_t
$$

对于 ODE，全部概率质量变化都必须由确定性速度场产生，因此速度场需要：

$$
-\frac12g^2s_t
$$

而反向 SDE 本身仍然具有随机扩散项。

反向时间中的随机扩散会继续影响概率密度，因此需要不同的漂移修正。

从这个角度看：

$$
\boxed{
\text{确定性运输}
+
\text{随机扩散}
}
$$

可以共同实现与纯确定性运输相同的边缘密度演化。

---

## 22. 一个具体例子：VP-SDE 的 Probability Flow ODE

VP-SDE：

$$
dX_t=
-\frac12\beta(t)X_tdt+
\sqrt{\beta(t)}dW_t
$$

因此：

$$
f(x,t)=-\frac12\beta(t)x
$$

$$
g(t)^2=\beta(t)
$$

所以 Probability Flow ODE：

$$
\boxed{
\frac{dX_t}{dt}
=
-\frac12\beta(t)X_t
-
\frac12\beta(t)\nabla\log p_t(X_t)
}
$$

即：

$$
\boxed{
\frac{dX_t}{dt}
=
-\frac12\beta(t)
\left[
X_t+\nabla\log p_t(X_t)
\right]
}
$$

如果使用网络：

$$
s_\theta(x,t)
$$

则：

$$
\boxed{
\frac{dX_t}{dt}
=
-\frac12\beta(t)
\left[
X_t+s_\theta(X_t,t)
\right]
}
$$

生成时，从：

$$
X_T\sim\mathcal N(0,I)
$$

出发，沿着：

$$
T\rightarrow0
$$

积分。

### 22.1 一个很有意思的特殊情况

假设：

$$
p_t=\mathcal N(0,I)
$$

那么：

$$
\nabla\log p_t(x)=-x
$$

代入：

$$
\frac{dX_t}{dt}
=
-\frac12\beta(t)[X_t-X_t]
$$

得到：

$$
\boxed{
\frac{dX_t}{dt}=0
}
$$

也就是说：

**如果整个分布已经是标准高斯，VP-SDE 对应的 Probability Flow ODE 可以让每个样本保持不动。**

但是 VP-SDE 本身仍然在随机运动：

$$
dX_t=-\frac12\beta X_tdt+\sqrt\beta dW_t
$$

这是一个非常好的例子。

它直接说明：

$$
\boxed{
\text{相同边缘分布}
\neq
\text{相同样本轨迹}
}
$$

---

# 第六部分：为什么 Probability Flow ODE 可以计算似然？

## 23. 从连续性方程推导瞬时变量变换公式

考虑 ODE：

$$
\boxed{
\frac{dX_t}{dt}=v_\theta(X_t,t)
}
$$

对应：

$$
\partial_tp_t=-\nabla\cdot(v_\theta p_t)
$$

我们希望计算沿 ODE 轨迹的：

$$
\log p_t(X_t)
$$

对时间求全导数：

$$
\frac{d}{dt}\log p_t(X_t)
=
\frac{\partial}{\partial t}\log p_t(X_t)
+
\nabla\log p_t(X_t)^\top\frac{dX_t}{dt}
$$

代入：

$$
\frac{dX_t}{dt}=v_\theta
$$

得到：

$$
\frac{d}{dt}\log p_t(X_t)
=
\frac{\partial}{\partial t}\log p_t(X_t)
+
\nabla\log p_t^\top v_\theta
$$

现在利用连续性方程：

$$
\partial_tp_t=-\nabla\cdot(v_\theta p_t)
$$

展开散度：

$$
\nabla\cdot(v_\theta p_t)
=
p_t\nabla\cdot v_\theta
+
v_\theta^\top\nabla p_t
$$

所以：

$$
\partial_tp_t
=
-p_t\nabla\cdot v_\theta
-
v_\theta^\top\nabla p_t
$$

两边除以：

$$
p_t
$$

得到：

$$
\frac{\partial}{\partial t}\log p_t
=
-\nabla\cdot v_\theta
-
v_\theta^\top\nabla\log p_t
$$

代回全导数：

$$
\begin{aligned}
\frac{d}{dt}\log p_t(X_t)
={}&
-\nabla\cdot v_\theta
-
v_\theta^\top\nabla\log p_t\\
&+
\nabla\log p_t^\top v_\theta
\end{aligned}
$$

最后两项抵消：

$$
\boxed{
\frac{d}{dt}\log p_t(X_t)
=
-\nabla\cdot v_\theta(X_t,t)
}
$$

这就是 Instantaneous Change of Variables Formula。

---

## 24. 积分得到数据似然

从：

$$
t=0
$$

积分到：

$$
t=T
$$

得到：

$$
\log p_T(X_T)-\log p_0(X_0)
=
-\int_0^T\nabla\cdot v_\theta(X_t,t)dt
$$

所以：

$$
\boxed{
\log p_0(X_0)
=
\log p_T(X_T)
+
\int_0^T\nabla\cdot v_\theta(X_t,t)dt
}
$$

这里：

$$
v_\theta(x,t)
=
f(x,t)-\frac12g(t)^2s_\theta(x,t)
$$

由于：

$$
p_T
$$

是已知先验，因此：

$$
\log p_T(X_T)
$$

可以直接计算。

例如：

$$
p_T=\mathcal N(0,I)
$$

则：

$$
\boxed{
\log p_T(X_T)
=
-\frac d2\log(2\pi)
-\frac12\|X_T\|^2
}
$$

因此，只要能计算：

$$
\int_0^T\nabla\cdot v_\theta(X_t,t)dt
$$

就能得到 ODE 模型对 $X_0$ 的对数密度。

**需要区分两种意义的“精确”：**

1. 从理论上说，Probability Flow ODE 定义了可通过变量变换计算密度的连续正规化流模型，不需要使用 ELBO 近似。
2. 实际计算仍然受到 ODE 数值积分误差、Score 网络误差和散度估计误差的影响。

因此，论文中的 exact likelihood 主要是指**直接计算 ODE 模型似然，而不是使用变分下界**，并不意味着有限计算精度下没有数值误差。

---

## 25. 为什么计算散度很昂贵？

散度定义为：

$$
\boxed{
\nabla\cdot v_\theta
=
\sum_{i=1}^{d}
\frac{\partial v_{\theta,i}}{\partial x_i}
}
$$

如果：

$$
x\in\mathbb R^d
$$

那么：

$$
v_\theta(x,t)\in\mathbb R^d
$$

其 Jacobian 为：

$$
J_\theta(x,t)
=
\frac{\partial v_\theta}{\partial x}
\in\mathbb R^{d\times d}
$$

散度就是 Jacobian 的迹：

$$
\boxed{
\nabla\cdot v_\theta
=
\operatorname{tr}(J_\theta)
}
$$

对于高维图像，直接构造整个 Jacobian 非常昂贵。

因此论文使用 Hutchinson Trace Estimator。

---

## 26. Hutchinson 散度估计的推导

设随机向量：

$$
z\in\mathbb R^d
$$

满足：

$$
\mathbb E[z]=0
$$

$$
\boxed{
\mathbb E[zz^\top]=I
}
$$

例如：

$$
z\sim\mathcal N(0,I)
$$

考虑：

$$
z^\top Jz
$$

展开：

$$
z^\top Jz
=
\sum_{i=1}^d\sum_{j=1}^d z_iJ_{ij}z_j
$$

取期望：

$$
\mathbb E[z^\top Jz]
=
\sum_{i,j}J_{ij}\mathbb E[z_iz_j]
$$

因为：

$$
\mathbb E[z_iz_j]=\delta_{ij}
$$

其中：

$$
\delta_{ij}
=
\begin{cases}
1,&i=j\\
0,&i\neq j
\end{cases}
$$

所以：

$$
\begin{aligned}
\mathbb E[z^\top Jz]
&=
\sum_iJ_{ii}\\
&=\operatorname{tr}(J)
\end{aligned}
$$

因此：

$$
\boxed{
\operatorname{tr}(J)
=
\mathbb E_z[z^\top Jz]
}
$$

也就是：

$$
\boxed{
\nabla\cdot v_\theta
=
\mathbb E_z
\left[
z^\top
\frac{\partial v_\theta}{\partial x}
z
\right]
}
$$

这样就不需要构造完整 Jacobian。

### 26.1 如何使用自动微分？

令：

$$
h(x)=z^\top v_\theta(x,t)
$$

对 $x$ 求梯度：

$$
\nabla_xh(x)=J_\theta^\top z
$$

再计算：

$$
z^\top\nabla_xh(x)
=
z^\top J_\theta^\top z
$$

因为标量满足：

$$
z^\top J_\theta^\top z=z^\top J_\theta z
$$

所以：

$$
\boxed{
z^\top\nabla_x[z^\top v_\theta(x,t)]
}
$$

就是散度的无偏随机估计。

这也是为什么论文可以在高维图像空间使用自动微分估计 ODE 似然。[arXiv](https://arxiv.org/html/2011.13456?utm_source=chatgpt.com)

---

# 第七部分：Predictor-Corrector 采样

## 27. Predictor 与 Corrector 分别解决什么问题？

论文提出 Predictor-Corrector（PC）框架。

假设当前样本：

$$
X_t
$$

我们希望得到：

$$
X_{t-h}
$$

其中：

$$
h>0
$$

**Predictor** 使用反向 SDE 的数值积分：

$$
X_t\rightarrow X_{t-h}
$$

但数值积分存在误差，因此生成样本的实际分布可能偏离：

$$
p_{t-h}
$$

**Corrector** 在固定时间：

$$
t-h
$$

使用 Langevin MCMC，尝试让样本分布更加接近目标边缘分布。

这两个过程的区别是：

$$
\boxed{
\text{Predictor：推进扩散时间}
}
$$

$$
\boxed{
\text{Corrector：固定扩散时间修正分布}
}
$$

---

## 28. Predictor 的推导

反向 SDE：

$$
dX_t=
[f(X_t,t)-g(t)^2s_\theta(X_t,t)]dt+
g(t)d\bar W_t
$$

令：

$$
dt=-h
$$

则：

$$
\boxed{
X_{t-h}
=
X_t+
h[g(t)^2s_\theta(X_t,t)-f(X_t,t)]
+
g(t)\sqrt h\,\epsilon
}
$$

这就是 Euler–Maruyama Predictor。

对于 VP-SDE：

$$
f(x,t)=-\frac12\beta(t)x
$$

$$
g(t)^2=\beta(t)
$$

所以：

$$
\boxed{
X_{t-h}
=
X_t+
h\left[
\frac12\beta(t)X_t+
\beta(t)s_\theta(X_t,t)
\right]
+
\sqrt{\beta(t)h}\epsilon
}
$$

对于 VE-SDE：

$$
f=0
$$

所以：

$$
\boxed{
X_{t-h}
=
X_t+
hg(t)^2s_\theta(X_t,t)
+
g(t)\sqrt h\,\epsilon
}
$$

---

## 29. Corrector 为什么使用 Langevin Dynamics？

在固定时间：

$$
t
$$

下，目标分布为：

$$
p_t(x)
$$

对应 Score：

$$
s_t(x)=\nabla\log p_t(x)
$$

考虑 Langevin SDE：

$$
\boxed{
dY_\tau=s_t(Y_\tau)d\tau+\sqrt2\,dW_\tau
}
$$

这里使用新的采样时间：

$$
\tau
$$

而不是扩散时间 $t$。

其 Fokker–Planck 方程：

$$
\partial_\tau q_\tau
=
-\nabla\cdot(q_\tau s_t)+\Delta q_\tau
$$

如果：

$$
q_\tau=p_t
$$

则：

$$
\begin{aligned}
\partial_\tau p_t
&=
-\nabla\cdot(p_t\nabla\log p_t)+\Delta p_t\\
&=
-\nabla\cdot(\nabla p_t)+\Delta p_t\\
&=0
\end{aligned}
$$

因此：

$$
\boxed{
p_t
}
$$

是这个 Langevin 动力学的平稳分布。

Euler 离散化：

$$
\boxed{
Y_{k+1}
=
Y_k+\eta s_\theta(Y_k,t)+\sqrt{2\eta}z_k
}
$$

其中：

$$
z_k\sim\mathcal N(0,I)
$$

这就是 Corrector。

需要注意，有限步未校正 Langevin 存在离散化偏差，因此 Corrector 并不保证每一步都严格改善分布误差；论文展示的是在相应实验设置下的经验改善。

---

## 30. Corrector 的自适应步长为什么是这个公式？

论文附录 G 中给出：

$$
\boxed{
\eta
=
2\left(
r\frac{\|z\|}{\|s_\theta(x,t)\|}
\right)^2
}
$$

其中：

$$
r>0
$$

是用户设置的信噪比参数。

这个公式并不是随意选择的。

从 Langevin 更新：

$$
\Delta x
=
\underbrace{\eta s_\theta(x,t)}_{\text{Score 漂移}}
+
\underbrace{\sqrt{2\eta}z}_{\text{随机噪声}}
$$

出发。

定义漂移与随机噪声的范数比值：

$$
\boxed{
r=
\frac{
\|\eta s_\theta(x,t)\|
}{
\|\sqrt{2\eta}z\|
}
}
$$

化简：

$$
r
=
\frac{
\eta\|s_\theta\|
}{
\sqrt{2\eta}\|z\|
}
$$

因为：

$$
\frac{\eta}{\sqrt{2\eta}}
=
\sqrt{\frac\eta2}
$$

所以：

$$
r
=
\sqrt{\frac\eta2}
\frac{\|s_\theta\|}{\|z\|}
$$

两边平方：

$$
r^2
=
\frac\eta2
\frac{\|s_\theta\|^2}{\|z\|^2}
$$

解出：

$$
\boxed{
\eta
=
2r^2
\frac{\|z\|^2}{\|s_\theta\|^2}
}
$$

即：

$$
\boxed{
\eta
=
2\left(
r\frac{\|z\|}{\|s_\theta\|}
\right)^2
}
$$

这就是论文中的自适应 Corrector 步长。

它的直观意义是：

**控制 Langevin 更新中 Score 漂移和随机探索的相对强度。**

当：

$$
\|s_\theta\|
$$

较大时，步长自动减小。

当：

$$
\|s_\theta\|
$$

较小时，步长可能增大。

实际实现中需要防止分母过小导致数值不稳定。

论文附录 G 给出了 VE 和 VP 的完整 PC 采样伪代码。[arXiv](https://arxiv.org/html/2011.13456?utm_source=chatgpt.com)

---

# 第八部分：附录中的广义 SDE

## 31. 为什么前面假设 $g(t)$ 不依赖状态？

正文主要讨论：

$$
dX_t=f(X_t,t)dt+g(t)dW_t
$$

其中：

$$
g(t)
$$

是标量，并且不依赖：

$$
X_t
$$

这种形式比较容易推导。

但论文附录 A 给出了更一般的 SDE：

$$
\boxed{
dX_t=f(X_t,t)dt+G(X_t,t)dW_t
}
\tag{15}
$$

其中：

$$
G(x,t)\in\mathbb R^{d\times d}
$$

是矩阵扩散系数。

定义扩散协方差矩阵：

$$
\boxed{
a(x,t)=G(x,t)G(x,t)^\top
}
$$

那么对应的 Fokker–Planck 方程为：

$$
\boxed{
\partial_tp_t
=
-\sum_i\partial_{x_i}(f_ip_t)
+
\frac12
\sum_{i,j}
\partial_{x_i}\partial_{x_j}(a_{ij}p_t)
}
$$

与标量扩散不同，这里的：

$$
a(x,t)
$$

可能依赖状态。

因此在推导反向 SDE 和 Probability Flow ODE 时，会出现额外的散度项。

---

## 32. 广义 Probability Flow ODE 的推导

从：

$$
\partial_tp_t
=
-\sum_i\partial_i(f_ip_t)
+
\frac12\sum_{i,j}\partial_i\partial_j(a_{ij}p_t)
$$

出发。

定义矩阵散度：

$$
\boxed{
(\nabla\cdot a)_i
=
\sum_j\partial_j a_{ij}
}
$$

利用乘积求导：

$$
\sum_j\partial_j(a_{ij}p_t)
=
\left(\sum_j\partial_j a_{ij}\right)p_t
+
\sum_j a_{ij}\partial_jp_t
$$

因为：

$$
\partial_jp_t=p_t\partial_j\log p_t
$$

所以：

$$
\sum_j\partial_j(a_{ij}p_t)
=
p_t(\nabla\cdot a)_i
+
p_t(a\nabla\log p_t)_i
$$

因此：

$$
\begin{aligned}
\partial_tp_t
=
-\nabla\cdot(fp_t)
+
\frac12\nabla\cdot
\left[
p_t\nabla\cdot a+
p_ta\nabla\log p_t
\right]
\end{aligned}
$$

整理为：

$$
\partial_tp_t
=
-\nabla\cdot
\left[
\left(
f-\frac12\nabla\cdot a
-\frac12a\nabla\log p_t
\right)p_t
\right]
$$

与连续性方程比较，得到：

$$
\boxed{
\frac{dX_t}{dt}
=
f(X_t,t)
-
\frac12\nabla\cdot a(X_t,t)
-
\frac12a(X_t,t)\nabla\log p_t(X_t)
}
\tag{17}
$$

其中：

$$
a=GG^\top
$$

这就是论文附录 A、D 中的广义 Probability Flow ODE。

当：

$$
G(x,t)=g(t)I
$$

时：

$$
a(x,t)=g(t)^2I
$$

由于不依赖 $x$：

$$
\nabla\cdot a=0
$$

于是退化为：

$$
\boxed{
\frac{dX_t}{dt}
=
f(X_t,t)-\frac12g(t)^2\nabla\log p_t(X_t)
}
$$

与正文完全一致。

---

## 33. 广义反向 SDE

论文附录 A 给出的反向 SDE 为：

$$
\boxed{
\begin{aligned}
dX_t
={}&
\left[
f(X_t,t)
-\nabla\cdot a(X_t,t)
-a(X_t,t)\nabla\log p_t(X_t)
\right]dt\\
&+
G(X_t,t)d\bar W_t
\end{aligned}
}
\tag{16}
$$

其中：

$$
dt<0
$$

与广义 Probability Flow ODE 对比：

$$
\boxed{
v_{\mathrm{ODE}}
=
f-\frac12\nabla\cdot a-\frac12a\nabla\log p_t
}
$$

可以发现：

- 反向 SDE 的扩散修正项系数为 $1$。
- Probability Flow ODE 的对应系数为 $\frac12$。

**特别注意：如果扩散矩阵依赖状态，不能直接使用简化公式 $f-g^2\nabla\log p_t$，否则会遗漏 $\nabla\cdot a$ 项。** [arXiv](https://arxiv.org/html/2011.13456?utm_source=chatgpt.com)

---

# 第九部分：条件生成与逆问题

## 34. 条件 Score 的推导

假设我们希望生成满足条件：

$$
y
$$

的数据。

目标分布为：

$$
p_t(x\mid y)
$$

根据贝叶斯公式：

$$
p_t(x\mid y)
=
\frac{
p_t(y\mid x)p_t(x)
}{
p(y)
}
$$

取对数：

$$
\log p_t(x\mid y)
=
\log p_t(x)+\log p_t(y\mid x)-\log p(y)
$$

对 $x$ 求梯度：

$$
\boxed{
\nabla_x\log p_t(x\mid y)
=
\nabla_x\log p_t(x)
+
\nabla_x\log p_t(y\mid x)
}
$$

因此：

$$
\boxed{
s_{\mathrm{conditional}}(x,t,y)
=
s_{\mathrm{unconditional}}(x,t)
+
\nabla_x\log p_t(y\mid x)
}
$$

将其代入反向 SDE：

$$
\boxed{
\begin{aligned}
dX_t
={}&
\Big[
f(X_t,t)
-g(t)^2s_\theta(X_t,t)\\
&\qquad
-g(t)^2\nabla_{X_t}\log p_t(y\mid X_t)
\Big]dt\\
&+
g(t)d\bar W_t
\end{aligned}
}
\tag{14}
$$

这是论文条件生成的重要公式。

---

## 35. 为什么条件似然必须与噪声时间匹配？

假设条件来自干净数据：

$$
y=h(x_0)
$$

例如：

- 图像类别。
- 图像缺失区域的已知像素。
- 灰度图像。
- 线性测量结果。

在反向扩散的中间时间：

$$
t>0
$$

我们手里只有噪声状态：

$$
x_t
$$

真正需要的是：

$$
\boxed{
p_t(y\mid x_t)
}
$$

而不是直接使用：

$$
p(y\mid x_0=x_t)
$$

因为：

$$
x_t
$$

并不是干净数据。

严格来说：

$$
\boxed{
p_t(y\mid x_t)
=
\int
p(y\mid x_0)
p(x_0\mid x_t)\,dx_0
}
$$

这是对未知干净数据 $x_0$ 的后验积分。

这个公式非常重要。

它说明：

**中间噪声状态下的条件引导，本质上需要考虑所有可能的干净数据及其后验概率。**

实际中这个积分通常很难直接计算，因此需要使用特殊条件结构、辅助模型或者近似方法。

---

## 36. 与自动驾驶碰撞引导的联系

假设你在轨迹规划中定义碰撞奖励：

$$
R(x_0,c)
$$

其中：

- $x_0$：干净规划轨迹。
- $c$：道路、障碍物、导航等场景条件。

希望生成：

$$
p_{\mathrm{target}}(x_0\mid c)
\propto
p_{\mathrm{base}}(x_0\mid c)
e^{\lambda R(x_0,c)}
$$

那么在噪声时间 $t$ 下，对应的目标边缘分布是：

$$
p_t^{\mathrm{target}}(x_t\mid c)
=
\int
p_{0t}(x_t\mid x_0)
p_{\mathrm{target}}(x_0\mid c)\,dx_0
$$

代入目标分布：

$$
p_t^{\mathrm{target}}(x_t\mid c)
\propto
\int
p_{0t}(x_t\mid x_0)
p_{\mathrm{base}}(x_0\mid c)
e^{\lambda R(x_0,c)}dx_0
$$

将基础边缘分布提出：

$$
\boxed{
p_t^{\mathrm{target}}(x_t\mid c)
\propto
p_t^{\mathrm{base}}(x_t\mid c)
\mathbb E_{\mathrm{base}}
\left[
e^{\lambda R(X_0,c)}
\mid X_t=x_t,c
\right]
}
$$

因此：

$$
\boxed{
\begin{aligned}
\nabla_{x_t}\log p_t^{\mathrm{target}}(x_t\mid c)
={}&
\nabla_{x_t}\log p_t^{\mathrm{base}}(x_t\mid c)\\
&+
\nabla_{x_t}\log
\mathbb E_{\mathrm{base}}
\left[
e^{\lambda R(X_0,c)}
\mid X_t=x_t,c
\right]
\end{aligned}
}
$$

这才是奖励指数倾斜分布对应的严格 Score 修正形式。

因此，直接计算：

$$
\nabla_{x_t}R(\hat x_0(x_t,t),c)
$$

通常只是一种近似引导方法。

它不自动等价于严格的条件 Score。

这与你之前研究的 Diffusion Planner 碰撞引导问题直接相关。

---

# 第十部分：论文的数值采样方法与实验

## 37. 三类采样器

论文主要讨论三种采样方式。

| 采样方法 | 动力学 | 是否随机 | 核心思想 |
|---|---|---|---|
| Reverse SDE | 反向随机微分方程 | 是 | 直接数值积分反向 SDE |
| Predictor-Corrector | 反向 SDE + Langevin MCMC | 是 | 推进时间并修正分布 |
| Probability Flow ODE | 确定性 ODE | 否（给定初始状态后） | 通过确定性速度场运输概率质量 |

### 37.1 Reverse Diffusion Sampler

论文附录 E 提出一种反向扩散离散化方法。

假设前向离散过程为：

$$
\boxed{
x_{i+1}=x_i+f_i(x_i)+G_iz_i
}
$$

其中：

$$
z_i\sim\mathcal N(0,I)
$$

那么对应的反向扩散离散化为：

$$
\boxed{
x_i
=
x_{i+1}
-
f_{i+1}(x_{i+1})
+
G_{i+1}G_{i+1}^\top
s_\theta(x_{i+1},i+1)
+
G_{i+1}z_{i+1}
}
$$

这就是附录 E 的公式 (46)。

其思想是：

**沿用前向过程的离散化结构，并根据反向 SDE 的漂移修正项构造对应的反向更新。**

这不意味着它在有限步长下是精确的反向转移分布；它仍然是一种数值近似。

---

## 38. 论文的网络结构改进

论文不仅提出理论框架，还改进了 NCSN 和 DDPM 的网络结构。

主要包括：

- 使用更强的 U-Net 风格主干。
- 引入 BigGAN 风格的残差块。
- 改进上采样与下采样结构。
- 在多尺度特征中使用注意力机制。
- 使用连续时间嵌入。
- 改进网络输出的尺度参数化。

作者将这些改进后的网络称为：

$$
\boxed{\text{NCSN++}}
$$

以及：

$$
\boxed{\text{DDPM++}}
$$

需要注意：

**NCSN++ 和 DDPM++ 不只是把原来的网络加深，它们还包含多种结构和训练设置上的改进。**

因此，论文中生成质量的提升并不能全部归因于 SDE 理论本身。

---

## 39. 论文的主要实验结论

论文在 CIFAR-10 上报告：

$$
\boxed{
\mathrm{FID}=2.20
}
$$

$$
\boxed{
\mathrm{IS}=9.89
}
$$

并在似然评估中报告：

$$
\boxed{
2.99\ \text{bits/dim}
}
$$

其中：

- 最佳 FID 来自 VE-SDE 与 NCSN++ 等设置的组合。
- 最佳似然来自 sub-VP-SDE 与 DDPM++ 等设置的组合。

它们不是同一个模型同时取得的结果。

论文还展示了高分辨率图像生成、图像修复、上色、类别条件生成和逆问题求解。

在采样器对比实验中，作者发现：

1. Reverse Diffusion Sampler 通常略优于对应的 Ancestral Sampling。
2. Predictor-Corrector 通常优于只使用相同数量级 Predictor 的方法。
3. Probability Flow ODE 可以使用自适应 ODE 求解器，并提供直接似然计算能力。
4. 不同 SDE 类型对 ODE 采样质量和似然表现具有明显影响。

这些结果支持了论文的核心观点：**统一 SDE 框架不仅解释了已有方法，也扩大了采样器和生成任务的设计空间。** [arxiv.org](https://arxiv.org/html/2011.13456?utm_source=chatgpt.com)

---

# 第十一部分：论文中几个特别重要的理论辨析

## 40. SDE 与 ODE 边缘分布相同，是否意味着可以任意互换？

不能简单这样理解。

理论上：

$$
p_t^{\mathrm{SDE}}=p_t^{\mathrm{ODE}}
$$

依赖以下条件：

- 两者使用同一个精确 Score。
- 两者的初始边缘分布一致。
- 对应的概率密度演化方程满足适当的唯一性条件。

实际中，网络只提供：

$$
s_\theta(x,t)\approx\nabla\log p_t(x)
$$

因此，使用同一个近似 Score 构造的 SDE 与 ODE，其实际边缘分布不一定严格相同。

此外，有限步长数值积分也会引入误差。

所以：

$$
\boxed{
\text{理论边缘一致}
\neq
\text{有限步采样结果完全一致}
}
$$

---

## 41. ODE 是否可以理解为 SDE 去掉随机项？

不可以。

如果直接从前向 SDE：

$$
dX_t=fdt+gdW_t
$$

删除随机项：

$$
dX_t=fdt
$$

那么通常会改变概率分布的演化。

正确的 Probability Flow ODE 是：

$$
\boxed{
dX_t=
\left(
f-\frac12g^2\nabla\log p_t
\right)dt
}
$$

其中新增的 Score 项正是为了补偿删除随机扩散项造成的概率密度变化。

---

## 42. Probability Flow ODE 的速度场是否唯一？

不一定。

假设某个速度场 $v$ 满足：

$$
\partial_tp_t=-\nabla\cdot(vp_t)
$$

如果存在另一个向量场 $u$，满足：

$$
\boxed{
\nabla\cdot(up_t)=0
}
$$

那么：

$$
v'=v+u
$$

也满足：

$$
\partial_tp_t=-\nabla\cdot(v'p_t)
$$

因为：

$$
\nabla\cdot(v'p_t)
=
\nabla\cdot(vp_t)+\nabla\cdot(up_t)
=
\nabla\cdot(vp_t)
$$

所以：

$$
\boxed{
\text{同一条概率分布路径可以对应多个速度场}
}
$$

论文给出的 Probability Flow ODE 是一个自然且具有明确 SDE 对应关系的选择，但并不意味着所有产生相同边缘分布的 ODE 都必须使用这一速度场。

这一点对后续理解 Flow Matching 特别重要。

---

## 43. 从 Flow Matching 的角度重新理解这篇论文

现在考虑 Flow Matching 中的连续性方程：

$$
\boxed{
\partial_tp_t=-\nabla\cdot(v_tp_t)
}
$$

Flow Matching 的目标是学习某个能够运输概率分布的速度场：

$$
v_t(x)
$$

而 Song 等人的 Probability Flow ODE 告诉我们：

对于给定的前向 SDE：

$$
dX_t=fdt+gdW_t
$$

可以构造：

$$
\boxed{
v_t(x)
=
f(x,t)-\frac12g(t)^2\nabla\log p_t(x)
}
$$

也就是说：

**给定 SDE 的漂移、扩散系数和边缘 Score，就能得到一个确定性概率流速度场。**

反过来，如果已知一个 ODE 速度场：

$$
v_t(x)
$$

以及它对应的边缘密度：

$$
p_t(x)
$$

我们也可以选择一个扩散系数：

$$
g(t)
$$

并定义前向 SDE 漂移：

$$
\boxed{
f(x,t)
=
v_t(x)+\frac12g(t)^2\nabla\log p_t(x)
}
$$

于是：

$$
\boxed{
dX_t=
\left[
v_t(X_t)
+
\frac12g(t)^2\nabla\log p_t(X_t)
\right]dt
+
g(t)dW_t
}
$$

这个 SDE 与原始 ODE 具有相同的边缘分布路径。

验证非常简单。

将：

$$
f=v+\frac12g^2\nabla\log p_t
$$

代入 Fokker–Planck 方程：

$$
\partial_tp_t
=
-\nabla\cdot(fp_t)+\frac12g^2\Delta p_t
$$

得到：

$$
\begin{aligned}
\partial_tp_t
={}&
-\nabla\cdot(vp_t)\\
&-
\frac12g^2\nabla\cdot(p_t\nabla\log p_t)\\
&+
\frac12g^2\Delta p_t
\end{aligned}
$$

因为：

$$
p_t\nabla\log p_t=\nabla p_t
$$

所以后两项抵消：

$$
\boxed{
\partial_tp_t=-\nabla\cdot(vp_t)
}
$$

因此，两者具有相同的边缘概率密度演化。

但这里有一个关键限制：

**构造这个等价 SDE，需要知道边缘 Score $\nabla\log p_t$。**

仅仅知道一个 Flow Matching 网络输出的速度：

$$
v_\theta(x,t)
$$

并不足以直接构造严格等价的 SDE。

这一点与你之前研究的 Flow Matching 强化学习探索问题直接相关。

---

# 第十二部分：完整总结

## 44. 三篇论文的发展脉络

| 论文 | 核心理论 | 训练目标 | 生成方法 |
|---|---|---|---|
| NCSN（2019） | Score Matching | 多尺度 DSM | Annealed Langevin |
| DDPM（2020） | ELBO、KL、DSM | 噪声预测 MSE | 反向马尔可夫链 |
| Score SDE（2021） | SDE、Fokker–Planck、时间反转 | 连续时间 DSM | Reverse SDE、PC、Probability Flow ODE |

三篇论文之间的联系可以概括为：

$$
\boxed{
\begin{aligned}
&\text{NCSN：学习 Score}\\
&\Downarrow\\
&\text{DDPM：学习反向去噪过程}\\
&\Downarrow\\
&\text{Song SDE：用连续随机过程统一两者}
\end{aligned}
}
$$

---

## 45. 整篇论文最重要的八个公式

**1. 前向 SDE**

$$
\boxed{
dX_t=f(X_t,t)dt+g(t)dW_t
}
$$

**2. Fokker–Planck 方程**

$$
\boxed{
\partial_tp_t
=
-\nabla\cdot(fp_t)+\frac12g^2\Delta p_t
}
$$

**3. VE-SDE**

$$
\boxed{
dX_t=
\sqrt{\frac{d\sigma^2(t)}{dt}}dW_t
}
$$

**4. VP-SDE**

$$
\boxed{
dX_t=
-\frac12\beta(t)X_tdt+\sqrt{\beta(t)}dW_t
}
$$

**5. 反向 SDE**

$$
\boxed{
dX_t=
[f(X_t,t)-g(t)^2\nabla\log p_t(X_t)]dt
+
g(t)d\bar W_t
}
$$

**6. 连续时间 DSM**

$$
\boxed{
\mathcal L
=
\mathbb E
\left[
\lambda(t)
\left\|
s_\theta(X_t,t)
-
\nabla\log p_{0t}(X_t\mid X_0)
\right\|^2
\right]
}
$$

**7. Probability Flow ODE**

$$
\boxed{
\frac{dX_t}{dt}
=
f(X_t,t)-\frac12g(t)^2\nabla\log p_t(X_t)
}
$$

**8. ODE 似然计算**

$$
\boxed{
\log p_0(X_0)
=
\log p_T(X_T)
+
\int_0^T\nabla\cdot v_\theta(X_t,t)dt
}
$$

---

## 46. 我认为这篇论文最值得深入理解的三个结论

**第一，Score 是连接随机扩散与反向生成的关键。**

前向 SDE 只需要设计：

$$
f(x,t),g(t)
$$

但反向 SDE 还需要：

$$
\nabla\log p_t(x)
$$

而 DSM 恰好提供了学习边缘 Score 的方法。

**第二，随机 SDE 和确定性 ODE 可以具有相同的边缘分布。**

这说明生成建模并不一定需要在反向过程中持续注入随机噪声。

可以通过确定性速度场完成概率质量运输。

但两者的轨迹、数值误差和实际采样效果仍然可能不同。

**第三，扩散模型与 Flow Matching 在连续性方程层面具有统一的数学基础。**

Score-based SDE 给出了：

$$
v_t=f-\frac12g^2\nabla\log p_t
$$

而 Flow Matching 直接学习：

$$
v_t
$$

因此，两者并不是毫无关系的两种生成模型，而是在概率分布演化层面存在明确联系。

**如果后续要把这篇论文真正掌握到可以推导和实现的程度，我建议下一步继续研究一个具体问题：在已知二维高斯混合分布真实 Score 的情况下，分别使用 Reverse SDE 和 Probability Flow ODE 采样，验证两者的边缘分布一致性，并通过散度积分计算 ODE 的对数似然。**

这个实验能直接验证整篇论文最核心的数学结论，也会为后续学习 DDIM、DPM-Solver 和 Flow Matching 打下基础。